# Sanity 1: Causal-Mask Jacobian Structure
Verifies block lower-triangular Jacobian structure and determinant factorization for token-wise causal flow.

In [ ]:
import torch
import matplotlib.pyplot as plt
from homeomorphism import jacobian

torch.manual_seed(0)
T, d = 3, 4
W = torch.tril(torch.randn(T * d, T * d) * 0.05)
W = W + torch.eye(T * d) * 0.5

def phi(h):
    x = h.reshape(-1)
    y = x + W @ x
    return y.reshape(T, d)

h = torch.randn(T, d, dtype=torch.float32)
bj, per_diag = jacobian.build_jacobian(phi, h, scope="full", evaluate="per_diagonal_slogdet")
rows = []
for i in range(T):
    row_blocks = [bj[(i, k)] for k in range(T)]
    rows.append(torch.cat(row_blocks, dim=1))
J = torch.cat(rows, dim=0)

upper_norm = torch.linalg.norm(torch.triu(J, diagonal=1)).item()
sign_full, logdet_full = torch.linalg.slogdet(J)
diag_log_sum = sum(float(v[1].item()) for v in per_diag.values())

print(f'Upper-triangle norm (should be near 0): {upper_norm:.3e}')
print(f'Full log|det|: {float(logdet_full):.6f}')
print(f'Sum diagonal-block log|det|: {diag_log_sum:.6f}')
print(f'Absolute diff: {abs(float(logdet_full) - diag_log_sum):.3e}')

plt.figure(figsize=(5, 4))
plt.imshow(J.detach().numpy(), cmap='coolwarm', aspect='auto')
plt.title('Full Jacobian Heatmap')
plt.colorbar()
plt.tight_layout()
plt.show()